### Main Skeleton - Orchestrator

In [ ]:
import fitz  # PyMuPDF
import os
import json
import re


def extract_text_from_pdf(pdf_path):
    """Opens a PDF and returns all text from all pages."""
    try:
        doc = fitz.open(pdf_path)
        full_text = ""
        for page in doc:
            full_text += page.get_text()
        doc.close()
        return full_text
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
        return None

def identify_bank(pdf_text):
    """
    Analyzes the text to identify which bank parser to use.
    """
    text_lower = pdf_text.lower()
    
    if "hdfc bank" in text_lower:
        return "HDFC"
    elif "icici bank" in text_lower:
        return "ICICI"
    elif "sbi card" in text_lower:
        return "SBI"
    elif "axis bank" in text_lower:
        return "AXIS"
    elif "kotak mahindra bank" in text_lower:
        return "KOTAK"
    else:
        return "UNKNOWN"

# --- helper functions ---

def clean_amount(amount_str):
    """Removes commas, '₹', and 'Rs.' from a string to get a clean number."""
    if not amount_str:
        return None
    return re.sub(r"[₹,]|Rs\.", "", amount_str).strip()

# --- Define one parser function for each bank ---

def parse_hdfc(pdf_text):
    """
    Extracts data specifically from an HDFC statement
    using the defined standard schema (v2 with transactions).
    """
    print("Parsing HDFC...")
    
    # Initialize all fields based on our new standard schema
    data = {
        "bank_name": "HDFC",
        "customer_name": None,
        "customer_address": None,
        "card_last_4": None,
        "card_type": None,
        "billing_cycle": None,
        "payment_due_date": None,
        "total_balance_due": None,
        "transactions": [] # Initialize as an empty list
    }
    
    # --- Regex Patterns for HDFC (Same as before) ---

    # 1. Name and Address (Robust)
    # We capture the entire block starting with a title and ending just before "Card Number:"
    name_addr_match = re.search(
        r"((?:Mr|Ms|Mrs)\..*?)\s*Card Number:", 
        pdf_text, 
        re.DOTALL
    )
    if name_addr_match:
        # Get the full block of text
        full_block = name_addr_match.group(1).strip()
        
        # Split by newlines and remove any empty lines
        lines = [line.strip() for line in full_block.split('\n') if line.strip()]
        
        if lines:
            data["customer_name"] = lines[0] # First line is the name
            if len(lines) > 1:
                # Join the rest of the lines as the address
                data["customer_address"] = " ".join(lines[1:])


    # 2. Card Last 4 Digits
    match = re.search(r"Card Number: XXXX XXXX XXXX\s*(\d{4})", pdf_text)
    if match:
        data["card_last_4"] = match.group(1)

    # 3. Card Type
    # Look for text between "Card Type:" and the next label, "Billing Cycle:"
    match = re.search(r"Card Type:\s*([\w\s]+)\s*Billing Cycle:", pdf_text)
    if match:
        data["card_type"] = match.group(1).strip()

    # 4. Billing Cycle
    match = re.search(r"Billing Cycle:\s*([\w\s,-]+)\s*Payment Due Date:", pdf_text, re.DOTALL)
    if match:
        data["billing_cycle"] = match.group(1).strip()

    # 5. Payment Due Date
    match = re.search(r"Payment Due Date:\s*([\w\s,]+)\s*Total Balance Due:", pdf_text, re.DOTALL)
    if match:
        data["payment_due_date"] = match.group(1).strip()

    # 6. Total Balance Due
    match = re.search(r"Total Balance Due:\s*₹\s*([\d,]+\.\d{2})", pdf_text)
    if match:
        data["total_balance_due"] = clean_amount(match.group(1))
        
    # --- NEW: Transaction Parsing ---
    
    # This regex finds all lines that match the transaction pattern:
    # Group 1: Date (e.g., "Oct 08, 2025")
    # Group 2: Description (any character, non-greedy)
    # Group 3: Amount (e.g., "12,500.00")
    transaction_pattern = re.compile(
        r"([A-Za-z]{3}\s\d{2},\s\d{4})\s+(.*?)\s+([\d,]+\.\d{2})"
    )
    
    # We use findall to get a list of all matches
    matches = transaction_pattern.findall(pdf_text)
    
    for match in matches:
        date, description, amount = match
        
        # A small check to make sure we don't grab the table header by mistake
        if description.lower() != "transaction details" and "total balance due:" not in description.lower():
            data["transactions"].append({
                "date": date.strip(),
                "description": description.strip(),
                "amount": clean_amount(amount)
            })
            
    return data


def parse_icici(pdf_text):
    """
    Extracts data specifically from an ICICI statement
    using the defined standard schema (v2 with transactions).
    """
    print("Parsing ICICI...")
    
    # 1. Initialize the standard schema dictionary
    data = {
        "bank_name": "ICICI",
        "customer_name": None,
        "customer_address": None,
        "card_last_4": None,
        "card_type": None,
        "billing_cycle": None,
        "payment_due_date": None,
        "total_balance_due": None,
        "transactions": []
    }

    # --- Regex Patterns for ICICI ---

    # 2. Name and Address (Robust)
    # We capture the entire block starting with a title and ending just before "Statement for Card:"
    name_addr_match = re.search(
        r"((?:Mr|Ms|Mrs)\..*?)\s*Statement for Card:", 
        pdf_text, 
        re.DOTALL
    )
    if name_addr_match:
        # Get the full block of text
        full_block = name_addr_match.group(1).strip()
        
        # Split by newlines and remove any empty lines
        lines = [line.strip() for line in full_block.split('\n') if line.strip()]
        
        if lines:
            data["customer_name"] = lines[0] # First line is the name
            if len(lines) > 1:
                # Join the rest of the lines as the address
                data["customer_address"] = " ".join(lines[1:])

    # 3. Card Last 4 Digits
    match = re.search(r"Statement for Card:\s*XXXX-(\d{4})", pdf_text)
    if match:
        data["card_last_4"] = match.group(1)

    # 4. Card Type
    # Look for the text between "Card:" and "Statement Period:"
    match = re.search(r"Card:\s*([\w\s]+)\s*Statement Period:", pdf_text, re.DOTALL)
    if match:
        data["card_type"] = match.group(1).strip()

    # 5. Billing Cycle
    match = re.search(r"Statement Period:\s*([\d/]+\s+to\s+[\d/]+)", pdf_text)
    if match:
        data["billing_cycle"] = match.group(1).strip()

    # 6. Payment Due Date
    # This is in the right-panel
    match = re.search(r"Due Date:\s*(\d{2}/\d{2}/\d{4})", pdf_text)
    if match:
        data["payment_due_date"] = match.group(1).strip()

    # 7. Total Balance Due
    # Also in the right-panel. Use re.DOTALL in case Rs. and the amount are on new lines
    match = re.search(r"Total Due:\s*Rs\.\s*([\d,]+\.\d{2})", pdf_text, re.DOTALL)
    if match:
        data["total_balance_due"] = clean_amount(match.group(1))

    # --- Transaction Parsing ---
    
    # Group 1: Date (dd/mm/yyyy)
    # Group 2: Description (any word/space characters, non-greedy)
    # Group 3: Amount
    transaction_pattern = re.compile(
        r"(\d{2}/\d{2}/\d{4})\s+([\w\s]+?)\s+([\d,]+\.\d{2})"
    )
    
    matches = transaction_pattern.findall(pdf_text)
    
    for match in matches:
        date, description, amount = match
        
        # Filter out the table header
        if description.lower().strip() != "description":
            data["transactions"].append({
                "date": date.strip(),
                "description": description.strip(),
                "amount": clean_amount(amount)
            })

    return data


def parse_sbi(pdf_text):
    """
    Extracts data specifically from an SBI statement
    using the defined standard schema (v2 with transactions).
    (Fix for partial/null date matches)
    """
    print("Parsing SBI...")
    
    # 1. Initialize the standard schema dictionary
    data = {
        "bank_name": "SBI",
        "customer_name": None,
        "customer_address": None,
        "card_last_4": None,
        "card_type": None,
        "billing_cycle": None,
        "payment_due_date": None,
        "total_balance_due": None,
        "transactions": []
    }

    # --- Regex Patterns for SBI ---

    # 2. Name and Address (Robust)
    name_addr_match = re.search(
        r"To,\s*(.*?)\s*Card Number", 
        pdf_text, 
        re.DOTALL
    )
    if name_addr_match:
        full_block = name_addr_match.group(1).strip()
        lines = [line.strip() for line in full_block.split('\n') if line.strip()]
        
        if lines:
            data["customer_name"] = lines[0]
            if len(lines) > 1:
                data["customer_address"] = " ".join(lines[1:])

    # 3. Card Last 4 Digits and Card Type
    match = re.search(r"Card Number\s*XXXX XXXX XXXX (\d{4})\s*\((.*?)\)", pdf_text, re.DOTALL)
    if match:
        data["card_last_4"] = match.group(1).strip()
        data["card_type"] = match.group(2).strip()

    # 4. Billing Cycle -- !! THIS IS THE FIX !!
    # Use the specific dd-Mon-yyyy format
    match = re.search(
        r"Billing Period\s*(\d{2}-[A-Za-z]{3}-\d{4}\s+to\s+\d{2}-[A-Za-z]{3}-\d{4})", 
        pdf_text, 
        re.DOTALL
    )
    if match:
        data["billing_cycle"] = match.group(1).strip()

    # 5. Payment Due Date -- !! THIS IS THE FIX !!
    # Use the specific dd-Mon-yyyy format
    match = re.search(
        r"Payment Due Date\s*(\d{2}-[A-Za-z]{3}-\d{4})", 
        pdf_text, 
        re.DOTALL
    )
    if match:
        data["payment_due_date"] = match.group(1).strip()

    # 6. Total Balance Due
    match = re.search(r"Total Amount Due\s*INR\s*([\d,]+\.\d{2})", pdf_text, re.DOTALL)
    if match:
        data["total_balance_due"] = clean_amount(match.group(1))

    # --- Transaction Parsing ---
    
    # This pattern was already specific and working, so we keep it
    transaction_pattern = re.compile(
        r"(\d{2}-[A-Za-z]{3}-\d{4})\s+(.*?)\s+([\d,]+\.\d{2})"
    )
    
    matches = transaction_pattern.findall(pdf_text)
    
    for match in matches:
        date, description, amount = match
        if description.lower().strip() not in ["description", "particulars"]:
            data["transactions"].append({
                "date": date.strip(),
                "description": description.strip(),
                "amount": clean_amount(amount)
            })

    return data


def parse_axis(pdf_text):
    """
    Extracts data specifically from an Axis statement
    using the defined standard schema (v2 with transactions).
    """
    print("Parsing Axis...")
    
    # 1. Initialize the standard schema dictionary
    data = {
        "bank_name": "AXIS",
        "customer_name": None,
        "customer_address": None,
        "card_last_4": None,
        "card_type": None,
        "billing_cycle": None,
        "payment_due_date": None,
        "total_balance_due": None,
        "transactions": []
    }

    # --- Regex Patterns for Axis ---

    # 2. Name and Address (Robust)
    # This template doesn't use "Mr." or "Ms."
    # We'll find the block between "Credit Card Statement" and "Card:"
    name_addr_match = re.search(
        r"Credit Card Statement\s*(.*?)\s*Card:", 
        pdf_text, 
        re.DOTALL
    )
    if name_addr_match:
        full_block = name_addr_match.group(1).strip()
        
        # Split by newlines and remove any empty lines
        lines = [line.strip() for line in full_block.split('\n') if line.strip()]
        
        if lines:
            data["customer_name"] = lines[0] # First line is the name
            if len(lines) > 1:
                # Join the rest of the lines as the address
                data["customer_address"] = " ".join(lines[1:])

    # 3. Card Last 4 Digits and Card Type
    # These are on the same line, in a different order
    match = re.search(
        r"Card:\s*([\w\s]+?)\s+ending in\s*(\d{4})", 
        pdf_text, 
        re.DOTALL
    )
    if match:
        data["card_type"] = match.group(1).strip()
        data["card_last_4"] = match.group(2).strip()

    # 4. Billing Cycle
    # This data is in the summary box
    match = re.search(
        r"Statement Period\s*([\d/]+\s*-\s*[\d/]+)", 
        pdf_text, 
        re.DOTALL
    )
    if match:
        data["billing_cycle"] = match.group(1).strip()

    # 5. Payment Due Date
    match = re.search(
        r"Payment Due Date\s*(\d{2}/\d{2}/\d{2})", 
        pdf_text, 
        re.DOTALL
    )
    if match:
        data["payment_due_date"] = match.group(1).strip()

    # 6. Total Balance Due
    match = re.search(
        r"Total Amount Due\s*₹\s*([\d,]+\.\d{2})", 
        pdf_text, 
        re.DOTALL
    )
    if match:
        data["total_balance_due"] = clean_amount(match.group(1))

    # --- Transaction Parsing ---
    
    # Group 1: Date (dd/mm/yy)
    # Group 2: Description (non-greedy)
    # Group 3: Amount
    transaction_pattern = re.compile(
        r"(\d{2}/\d{2}/\d{2})\s+(.*?)\s+([\d,]+\.\d{2})"
    )
    
    matches = transaction_pattern.findall(pdf_text)
    
    for match in matches:
        date, description, amount = match
        
        # Filter out the table header
        if description.lower().strip() != "particulars":
            data["transactions"].append({
                "date": date.strip(),
                "description": description.strip(),
                "amount": clean_amount(amount)
            })

    return data


def parse_kotak(pdf_text):
    """
    Extracts data specifically from a Kotak statement
    using the defined standard schema (v2 with transactions).
    (Fix for parsing name/address block)
    """
    print("Parsing Kotak...")
    
    # 1. Initialize the standard schema dictionary
    data = {
        "bank_name": "KOTAK",
        "customer_name": None,
        "customer_address": None,
        "card_last_4": None,
        "card_type": None,
        "billing_cycle": None,
        "payment_due_date": None,
        "total_balance_due": None,
        "transactions": []
    }

    # --- Regex Patterns for Kotak ---

    # 2. Name and Address (Robust) -- !! THIS IS THE FIX !!
    # Find the text block BETWEEN the first line of stars and the first line of dashes
    name_addr_match = re.search(
        r"\*{10,}\s*\n(.*?)\n\s*-{10,}", 
        pdf_text, 
        re.DOTALL
    )
    
    if name_addr_match:
        full_block = name_addr_match.group(1).strip()
        
        # Split by newlines and remove any empty lines
        all_lines = [line.strip() for line in full_block.split('\n') if line.strip()]
        
        # --- This is the new logic ---
        # Filter out the header lines (those starting with '*')
        content_lines = [line for line in all_lines if not line.startswith('*')]
        # --- End of new logic ---
        
        if content_lines:
            data["customer_name"] = content_lines[0] # First "real" line is the name
            if len(content_lines) > 1:
                # Join the rest of the "real" lines as the address
                data["customer_address"] = " ".join(content_lines[1:])

    # 3. Card Type
    match = re.search(r"Card Type\s*:\s*(.*?)\s*Card Number", pdf_text, re.DOTALL)
    if match:
        data["card_type"] = match.group(1).strip()

    # 4. Card Last 4 Digits (This was fixed and working)
    match = re.search(r"Card Number\s*:\s*.*?(\d{4})", pdf_text, re.DOTALL)
    if match:
        data["card_last_4"] = match.group(1).strip()

    # 5. Billing Cycle
    match = re.search(r"Statement Period\s*:\s*([\d/]+\s*-\s*[\d/]+)", pdf_text, re.DOTALL)
    if match:
        data["billing_cycle"] = match.group(1).strip()

    # 6. Payment Due Date
    match = re.search(r"Payment Due Date\s*:\s*(\d{2}/\d{2}/\d{4})", pdf_text, re.DOTALL)
    if match:
        data["payment_due_date"] = match.group(1).strip()

    # 7. Total Balance Due
    match = re.search(r"Total Amount Due\s*:\s*Rs\.\s*([\d,]+\.\d{2})", pdf_text, re.DOTALL)
    if match:
        data["total_balance_due"] = clean_amount(match.group(1))

    # --- Transaction Parsing (This was working) ---
    transaction_pattern = re.compile(
        r"(\d{2}-[A-Z]{3}-\d{2})\s+(.*?)\s+([\d,]+\.\d{2})"
    )
    
    matches = transaction_pattern.findall(pdf_text)
    
    for match in matches:
        date, description, amount = match
        if description.lower().strip() != "description":
            data["transactions"].append({
                "date": date.strip(),
                "description": description.strip(),
                "amount": clean_amount(amount)
            })

    return data

In [39]:
# --- The Orchestrator (Main execution) ---

def main():
    # A mapping from our bank "ID" to the function that parses it
    PARSER_MAP = {
        "HDFC": parse_hdfc,
        "ICICI": parse_icici,
        "SBI": parse_sbi,
        "AXIS": parse_axis,
        "KOTAK": parse_kotak,
    }
    
    pdf_files = ["data\d1.pdf", "data\d2.pdf", "data\d3.pdf", "data\d4.pdf", "data\d5.pdf"]
    all_extracted_data = []

    for pdf_file in pdf_files:
        print(f"--- Processing {pdf_file} ---")
        
        # 1. Extract Text
        text = extract_text_from_pdf(pdf_file)
        if not text:
            continue
            
        # 2. Identify Bank (The "Router")
        bank_id = identify_bank(text)
        
        # 3. Route to correct parser
        if bank_id in PARSER_MAP:
            parser_function = PARSER_MAP[bank_id]
            extracted_data = parser_function(text)
            all_extracted_data.append(extracted_data)
        else:
            print(f"Could not identify bank for {pdf_file}. Skipping.")
            
    # 4. Save all data to JSON
    with open("all_statements.json", "w") as f:
        json.dump(all_extracted_data, f, indent=2)
        
    print("\nDone. Extracted data saved to 'all_statements.json'")

if __name__ == "__main__":
    main()

--- Processing data\d1.pdf ---
Parsing HDFC...
--- Processing data\d2.pdf ---
Parsing ICICI...
--- Processing data\d3.pdf ---
Parsing SBI...
--- Processing data\d4.pdf ---
Parsing Axis...
--- Processing data\d5.pdf ---
Parsing Kotak...

Done. Extracted data saved to 'all_statements.json'


In [33]:
print(extract_text_from_pdf("data/d5.pdf"))

KOTAK MAHINDRA BANK - CREDIT CARD
*************************************************************************
** MONTHLY STATEMENT                         **
*************************************************************************
VIKRAM CHOPRA
77, ALIPORE ROAD
KOLKATA, WB 700027
-------------------------------------------------------------------------
SUMMARY OF ACCOUNT
-------------------------------------------------------------------------
Card Type             : 
    Kotak Urbane Gold Visa
Card Number           : 
    XXXX XXXX XXXX 
    4422
Statement Generation  : 10 Nov 2025
Statement Period      : 
    10/10/2025 - 09/11/2025
Payment Due Date      : 
    28/11/2025
Total Amount Due      : 
    Rs. 9,876.50
Minimum Amount Due    : Rs. 493.83
-------------------------------------------------------------------------
TRANSACTION DETAILS
-------------------------------------------------------------------------
DATE          DESCRIPTION                         AMOUNT (Rs.)
----------